In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import cv2
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [3]:
categories = ["bottle", "capsule", "hazelnut", "metal_nut"]

In [4]:
DATA_ROOT = "C:\\Users\\ADMIN\\Documents\\CV_Proj\\Data\\mvtec"

In [5]:
import os

# project root (go one level up from notebooks)
BASE_DIR = os.path.abspath("..")

MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "Results")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Models path:", MODELS_DIR)
print("Results path:", RESULTS_DIR)

Models path: c:\Users\ADMIN\Documents\CV_Proj\models
Results path: c:\Users\ADMIN\Documents\CV_Proj\Results


In [6]:
transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor()
])

In [7]:
class MVTecDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = root_dir
        self.image_paths = [
            os.path.join(root_dir, img)
            for img in os.listdir(root_dir)
        ]

        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image

In [8]:
class Autoencoder(nn.Module):

    def __init__(self):
        super(Autoencoder, self).__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):

        x = self.encoder(x)
        x = self.decoder(x)

        return x

In [9]:
def train_autoencoder(category):

    print(f"\nTraining model for {category}")

    train_path = os.path.join(DATA_ROOT, category, "train/good")

    dataset = MVTecDataset(train_path, transform)

    dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

    model = Autoencoder().to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    epoch_losses = []

    for epoch in range(20):

        running_loss = 0

        for images in tqdm(dataloader):

            images = images.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, images)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(dataloader)

        epoch_losses.append(epoch_loss)

        print(f"{category} Epoch {epoch+1}: {epoch_loss}")

    return model, epoch_losses

In [10]:
def save_results(category, model, epoch_losses):

    model_path = os.path.join(MODELS_DIR, f"{category}_autoencoder.pth")

    torch.save(model.state_dict(), model_path)

    plt.figure(figsize=(10,5))

    plt.plot(epoch_losses, linewidth=2)

    plt.title(f"{category} Training Loss")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")

    plt.grid(True)

    graph_path = os.path.join(RESULTS_DIR, f"{category}_loss.png")

    plt.savefig(graph_path)

    plt.close()

In [11]:
trained_models = {}

for category in categories:

    model, losses = train_autoencoder(category)

    save_results(category, model, losses)

    trained_models[category] = model


Training model for bottle


100%|██████████| 14/14 [00:08<00:00,  1.71it/s]


bottle Epoch 1: 0.12045047485402652


100%|██████████| 14/14 [00:03<00:00,  3.67it/s]


bottle Epoch 2: 0.05633426271378994


...

metal_nut Epoch 18: 0.001070318211402212


100%|██████████| 14/14 [00:03<00:00,  4.01it/s]


metal_nut Epoch 19: 0.0010579144623729267


100%|██████████| 14/14 [00:03<00:00,  4.44it/s]

metal_nut Epoch 20: 0.0009860928569521224


In [12]:
print(os.getcwd())

c:\Users\ADMIN\Documents\CV_Proj\notebooks


In [13]:
def generate_anomaly_map(model, image_path):

    model.eval()

    img = Image.open(image_path).convert("RGB")

    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        reconstruction = model(img_tensor)

    error = torch.abs(img_tensor - reconstruction)

    error_map = torch.mean(error, dim=1).squeeze().cpu().numpy()

    return img, error_map

In [14]:
def smooth_anomaly_map(anomaly_map):

    smoothed = cv2.GaussianBlur(anomaly_map, (11,11), 0)

    return smoothed

In [15]:
def create_defect_mask(anomaly_map, pixel_threshold=0.02):

    mask = anomaly_map > pixel_threshold

    return mask

In [16]:
def overlay_mask(original_img, mask):

    img = np.array(original_img)

    mask_resized = cv2.resize(mask.astype(np.uint8),
                              (img.shape[1], img.shape[0]))

    mask_resized = mask_resized.astype(bool)

    img[mask_resized] = [255, 0, 0]

    return img

In [17]:
def run_detection(category, model):

    print(f"\nRunning anomaly detection for {category}")

    test_dir = os.path.join(DATA_ROOT, category, "test")

    save_dir = os.path.join(RESULTS_DIR, category)

    os.makedirs(save_dir, exist_ok=True)

    for defect_type in os.listdir(test_dir):

        folder = os.path.join(test_dir, defect_type)

        if not os.path.isdir(folder):
            continue

        for img_name in os.listdir(folder)[:5]:

            img_path = os.path.join(folder, img_name)

            original, anomaly_map = generate_anomaly_map(model, img_path)

            smoothed = smooth_anomaly_map(anomaly_map)

            mask = create_defect_mask(smoothed)

            overlay = overlay_mask(original, mask)

            save_path = os.path.join(save_dir, f"{defect_type}_{img_name}")

            plt.imsave(save_path, overlay)

In [18]:
trained_models = {}

for category in categories:

    model, losses = train_autoencoder(category)

    save_results(category, model, losses)

    run_detection(category, model)

    trained_models[category] = model


Training model for bottle


100%|██████████| 14/14 [00:03<00:00,  3.65it/s]


bottle Epoch 1: 0.11306584041033473


100%|██████████| 14/14 [00:03<00:00,  3.84it/s]


bottle Epoch 2: 0.05193398813051837


...

metal_nut Epoch 18: 0.0013192615505041821


100%|██████████| 14/14 [00:03<00:00,  4.43it/s]


metal_nut Epoch 19: 0.0012594991962292365


100%|██████████| 14/14 [00:03<00:00,  3.93it/s]


metal_nut Epoch 20: 0.0014498103243697966

Running anomaly detection for metal_nut


In [19]:
from patchcore.feature_extractor import ResNetFeatureExtractor

ModuleNotFoundError: No module named 'patchcore'

In [20]:
from patchcore.feature_extractor import ResNetFeatureExtractor

extractor = ResNetFeatureExtractor().to(device)

dummy = torch.randn(1,3,256,256).to(device)

features = extractor(dummy)

print(features.shape)

ModuleNotFoundError: No module named 'patchcore'